# 02｜ETF分红、份额折算、六宽基组合（研报第17—18页）

**先运行Notebook 01！** 本Notebook从 `outputs/index_signals/` 读取6只指数产生的交易仓位信号，之后把信号映射到6只ETF，计算含现金分红和份额调整的历史模拟收益。

**与研报的边界：** 这是*固定六只ETF的简化复现*，不是严格复现研究中的历史动态备选池。省略历史20亿元基金规模筛选、60日平均成交额筛选的动态ETF选择及短融ETF收益。无风险现金收益先假设0。回测终点固定2025-09-30，忽略其后行情、分红。

每个代码单元格只做一个小任务。代码不访问网络。

## 第1章：读取ETF行情和指数信号

确认目录内已放入你上传的6只ETF CSV和4只分红CSV。159915、588000的分红按你所确认“无分红”处理（仅在本回测范围内采用该假设）。

In [ ]:
# 1. 导入必要的库
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
# 2. 定义数据目录和时间、交易成本假设
ROOT = Path.cwd()
ETF_DIR = ROOT/'data/raw/etf'
DIV_DIR = ROOT/'data/raw/dividends'
SIGNAL_DIR = ROOT/'outputs/index_signals'
OUT = ROOT/'outputs'
START, END = pd.Timestamp('2019-01-02'),pd.Timestamp('2025-09-30')
FEE_BPS = 0.0  # 每单位单边换手成本，先设为0（假设）

In [ ]:
# 3. 建立指数与ETF的一一对应关系
INDEX_ETF = {
    '000016':'510050','000300':'510300','000905':'510500',
    '000852':'512100','399006':'159915','000688':'588000'
}
print(INDEX_ETF)

In [ ]:
# 4. 检查我们需要的原始文件是否齐全
for code in INDEX_ETF.values():
    price = ETF_DIR/f'{code}_etf.csv'
    div = DIV_DIR/f'{code}_dividend.csv'
    print(code,'ETF行情:',price.exists(),'分红表:',div.exists())

In [ ]:
# 5. 读取6只ETF价格（文件中已有英文列名）
etfs = {}
for code in INDEX_ETF.values():
    d = pd.read_csv(ETF_DIR/f'{code}_etf.csv',dtype={'etf_code':str})
    d['date'] = pd.to_datetime(d['date'])
    d = d.loc[d.date<=END].sort_values('date').set_index('date')
    assert d.index.is_unique and d['close'].notna().all()
    etfs[code] = d
print('ETF行情读取完成:',len(etfs))

In [ ]:
# 6. 查看ETF上市行情起点和末日（不是基金合同成立日）
coverage = pd.DataFrame([{'ETF':c,'交易首日':str(d.index.min().date()),
                          '最后一日':str(d.index.max().date()),'行数':len(d)} for c,d in etfs.items()])
display(coverage)

In [ ]:
# 7. 读取Notebook 01保存的6只指数交易信号
missing = [code for code in INDEX_ETF if not (SIGNAL_DIR/f'{code}.csv').exists()]
if missing:
    raise FileNotFoundError(f'缺少指数信号: {missing}。先在Notebook 01运行第34格，并放齐6份指数CSV。')
signals = {code:pd.read_csv(SIGNAL_DIR/f'{code}.csv',parse_dates=['date']).set_index('date')
           for code in INDEX_ETF}
print('已读取指数信号:',len(signals))

## 第2章：把累计分红变成每次分红

你上传的 `日期、累计分红` 是累加值，不能把累计值全部作为当日分红。例如510300在2017-01-23累计0.222，上一条0.167，因此单次分红是0.055元/份。

特别提醒：分红CSV里可能有**累计分红为0的份额折算记录**，那不是现金分红。我们仅保留差分后大于0的事件。

In [ ]:
# 8. 读取一只ETF的历史累计分红表
def read_dividend(code):
    file = DIV_DIR/f'{code}_dividend.csv'
    if not file.exists():
        if code in {'159915','588000'}: return pd.Series(dtype=float)
        raise FileNotFoundError(f'缺少分红文件: {file}')
    d = pd.read_csv(file)
    d['日期'] = pd.to_datetime(d['日期'])
    d = d.sort_values('日期')
    d['本次分红'] = d['累计分红'].diff().fillna(d['累计分红'])
    assert (d['本次分红']>=-1e-8).all(), f'{code}累计分红数据存在回退，请核对'
    return d.set_index('日期')['本次分红'].loc[lambda x:x>1e-8]

In [ ]:
# 9. 先查看510300完整分红明细
etf_div_300 = read_dividend('510300')
display(etf_div_300.to_frame('本次分红').tail(10))

In [ ]:
# 10. 计算6只ETF的分红事件表
all_div = {code:read_dividend(code) for code in INDEX_ETF.values()}
print({code:len(s.loc[:END]) for code,s in all_div.items()})

## 第3章：处理ETF份额折算（非常关键）

- **510500**：2015-04-15旧1份折算为约0.2803新份；2022-08-26份额分拆1:1.1454，2022-08-29在新份额口径下复牌交易。
- **512100**：2022-09-02旧1份合并为0.36555新份，2022-09-05恢复交易。

这意味着**分拆/合并前后要比较“投资者持有份额的总价值”，不能把新份额单价与旧份额单价直接求涨跌幅。**

参考资料：[510500分红与折算记录](https://fundf10.eastmoney.com/fhsp_510500.html)、[512100交易所基金份额合并公告](https://www.sse.com.cn/disclosure/fund/announcement/c/new/2022-08-24/512100_20220824_1_x5QccCZk.pdf)。

In [ ]:
# 11. 记录份额变化：每1旧份额变成多少新份额
# 日期选价格首次反映新份额的交易日
SHARE_CHANGES = {
    '510500': {'2015-04-15':0.2803, '2022-08-29':1.1454},
    '512100': {'2022-09-05':0.36555}
}
print('需要处理的份额折算:',SHARE_CHANGES)

In [ ]:
# 12. 看512100原始价格跳变（尚未调整）
d = etfs['512100']['close']
raw_return = d.pct_change()
display(pd.DataFrame({'close':d,'未调整收益':raw_return}).loc['2022-09-01':'2022-09-06'])

In [ ]:
# 13. 为ETF逐日创建份额换算系数
def share_ratio(code, dates):
    factor = pd.Series(1.0,index=dates)
    for date,value in SHARE_CHANGES.get(code,{}).items():
        day = pd.Timestamp(date)
        if day in factor.index: factor.loc[day] = value
        elif day <= END: raise ValueError(f'{code}缺少折算后交易日{date}')
    return factor

In [ ]:
# 14. 为一只ETF计算含分红、折算的近似每日总收益
def total_return(code):
    price = etfs[code]['close'].astype(float)
    dividend = all_div[code].reindex(price.index,fill_value=0)
    ratio = share_ratio(code,price.index)
    return ((price*ratio+dividend)/price.shift(1)-1).rename(code)

In [ ]:
# 15. 检验512100合并日收益已正常化
adjusted = total_return('512100')
display(pd.DataFrame({'原始涨跌幅':raw_return,'修正后涨跌幅':adjusted}).loc['2022-09-01':'2022-09-06'])

In [ ]:
# 16. 再检验510500在2022-08-29的分拆日
d = etfs['510500']['close']
check = pd.DataFrame({'原始涨跌幅':d.pct_change(),'修正后涨跌幅':total_return('510500')})
display(check.loc['2022-08-25':'2022-08-30'])

In [ ]:
# 17. 汇总6只ETF的每日总收益序列
returns = pd.concat([total_return(code) for code in INDEX_ETF.values()],axis=1).sort_index()
returns = returns.loc[:END]
display(returns.tail())

In [ ]:
# 18. 做一个异常收益提示（只检查，不自动改数据）
alerts = (returns.loc[START:END].abs()>0.35).sum()
print('2019—2025年：修正后单日收益绝对值超过35%的天数')
display(alerts.to_frame('异常天数'))

**收益口径提醒：**这里把分红视为除息日收到并立即参与持有期总收益，这是一种近似总回报处理。实际现金派发通常晚于除息日；严格现金账本应跟踪到账日和份额。若发生同一交易日同时分红与份额折算，要核对每份分红适用的份额口径。当前上传样本中，上述折算日没有非零现金分红。

## 第4章：建立六宽基ETF组合（对应第17—18页）

研报核心逻辑：**指数计算网格信号 → 映射至ETF → 组合配置**。这里只采用你已下载的6只固定ETF，并且仅在该ETF已经上市、跟踪指数信号已可用时纳入。

每只ETF的基础配置上限按照当日有效ETF数量N取 `1/N`；网格仓位是对这一上限的使用比例。我们默认使用第9页的**择时策略2 `w_t2`**。

In [ ]:
# 19. 建立统一回测交易日历
calendar = returns.loc[START:END].index
print('回测交易日数量:',len(calendar), '区间:',calendar.min().date(),calendar.max().date())

In [ ]:
# 20. 判断当天某只ETF是否已上市且有指数交易信号
eligible = pd.DataFrame(index=calendar)
for idx,etf in INDEX_ETF.items():
    listed = calendar>=etfs[etf].index.min()
    ready = signals[idx]['w_t2'].reindex(calendar).notna()
    eligible[etf] = listed & ready
print('每年的有效ETF数量示例:');display(eligible.sum(axis=1).resample('YE').last())

In [ ]:
# 21. 计算当天的ETF数量N及单只ETF权重上限1/N
n_eligible = eligible.sum(axis=1)
limit = 1/n_eligible.replace(0,np.nan)
display(pd.DataFrame({'N':n_eligible,'每只ETF上限':limit}).loc['2020-11-10':'2020-11-20'])

In [ ]:
# 22. 根据指数择时策略2计算每只ETF的目标权重
target = pd.DataFrame(index=calendar)
for idx,etf in INDEX_ETF.items():
    signal = signals[idx]['w_t2'].reindex(calendar).fillna(0)
    target[etf] = (eligible[etf]*signal*limit).fillna(0)
assert (target.sum(axis=1)<=1+1e-9).all()
display(target.tail())

In [ ]:
# 23. 处理信号和执行的两交易日滞后
# t日收盘知道目标; t+1日收盘执行; t+2日收益才归属新仓位
holdings = target.shift(2).fillna(0)
executed = target.shift(1).fillna(0)
display(holdings.tail())

In [ ]:
# 24. 计算每日换手与模拟交易费用
turnover = executed.diff().abs().sum(axis=1)
turnover.iloc[0] = executed.iloc[0].abs().sum()
trading_cost = turnover*FEE_BPS/10000
print('合计单边换手量:',round(turnover.sum(),3))

In [ ]:
# 25. 汇总ETF组合的每日收益（闲置资金收益暂按0）
r_on_calendar = returns.reindex(calendar).fillna(0)
portfolio_return = (holdings*r_on_calendar).sum(axis=1)-trading_cost
portfolio_nav = (1+portfolio_return).cumprod()
print('最终净值:',round(portfolio_nav.iloc[-1],4))

In [ ]:
# 26. 画ETF组合净值
portfolio_nav.plot(figsize=(11,4),title='Six broad-based ETF grid portfolio (simplified)')
plt.grid(alpha=.2);plt.show()

In [ ]:
# 27. 画组合整体权益仓位
holdings.sum(axis=1).plot(figsize=(11,3),title='Portfolio equity allocation')
plt.ylim(0,1);plt.grid(alpha=.2);plt.show()

In [ ]:
# 28. 计算组合的年度收益表
annual = (1+portfolio_return).groupby(portfolio_return.index.year).prod()-1
display(annual.to_frame('年度收益').style.format('{:.2%}'))

In [ ]:
# 29. 计算年化收益、最大回撤、年化波动率
vol = portfolio_return.std()*np.sqrt(252)
peaks = np.maximum.accumulate(np.r_[1.,portfolio_nav.to_numpy()])
metrics = pd.Series({'年化收益':portfolio_nav.iloc[-1]**(252/len(portfolio_return))-1,
                     '最大回撤':np.max(1-np.r_[1.,portfolio_nav.to_numpy()]/peaks),
                     '年化波动':vol})
display(metrics.to_frame('本次简化复现').style.format('{:.2%}'))

In [ ]:
# 30. 放在研报第18页数字旁做对照（口径不一致）
report = pd.Series({'年化收益':0.1178,'最大回撤':0.1150,'年化波动':0.0868})
comparison = pd.DataFrame({'广发研报六宽基组合':report,'你的固定六ETF近似组合':metrics})
display(comparison.style.format('{:.2%}'))

**为什么不应要求两列一致？** 你采用固定6只ETF，未按照历史规模/流动性动态选择，现金部分按零收益而非短融ETF，交易时点与ATR/N1窗口是实验假设，且分红日采用简化总收益处理。研报第18页披露的结果是更完整的历史回测。

## 第5章：导出图表与表格（用于写复现报告）

In [ ]:
# 31. 保存每日报酬、净值和权益仓位
out = pd.DataFrame({'daily_return':portfolio_return,
                    'nav':portfolio_nav,'equity_weight':holdings.sum(axis=1),
                    'turnover':turnover})
out.to_csv(OUT/'portfolio_daily.csv',encoding='utf-8-sig',index_label='date')
print('保存完成:',OUT/'portfolio_daily.csv')

In [ ]:
# 32. 保存ETF持仓、年度收益与绩效表
holdings.to_csv(OUT/'portfolio_etf_weights.csv',encoding='utf-8-sig',index_label='date')
annual.to_frame('return').to_csv(OUT/'portfolio_annual_returns.csv',encoding='utf-8-sig')
comparison.to_csv(OUT/'portfolio_vs_report.csv',encoding='utf-8-sig')
print('CSV表格已保存')

In [ ]:
# 33. 保存最终净值图片
ax = portfolio_nav.plot(figsize=(11,4),title='Six broad-based ETF portfolio')
ax.grid(alpha=.2)
plt.tight_layout()
plt.savefig(OUT/'portfolio_nav.png',dpi=160)
plt.show()

### 完成后你就有了这些研究成果

- 研报第6页：ATR、NATR及相应指标图。
- 研报第7—8页：基础网格目标仓位、净值对比。
- 研报第9—10页：4种ATR择时规则及收益比较。
- 研报第17—18页：**固定六宽基ETF近似组合**的仓位、年度收益、净值、回撤和与研报结果的口径差异。

### 建议写入报告的限制

1. ATR窗口N与趋势窗口N1均先取20（实验假设）；交易时点使用两交易日滞后（实验假设）。
2. ETF样本固定，未还原研报的每日历史规模、流动性筛选；现金收益设为0。
3. 分红表来自用户上传累计分红，159915与588000按用户说明无分红；510500及512100已处理明确的份额折算。
4. 2022年9月2日等ETF停牌或原始行情缺失期间，本教学版对缺失当日ETF收益暂按0处理，**未建立逐笔可成交性的精细仿真**。
5. 六宽基组合复现值不保证接近研报的11.78%年化、11.50%最大回撤和8.68%年化波动率。